# Time-Varying Covariates in Survival Analysis

## Overview

Standard Cox regression assumes covariates are measured once at baseline and remain constant throughout follow-up. Many real covariates change over time — treatment dose, water quality measurements, disease status, habitat condition. Ignoring this produces biased estimates.

**Two distinct uses of time-varying covariates:**

| Use | Meaning | Typical application |
|---|---|---|
| **External time-varying covariate** | Covariate value changes but is independent of the subject's survival process | Seasonal temperature, policy changes, environmental measurements |
| **Internal time-varying covariate** | Covariate reflects the subject's own biological/behavioural state | Biomarker values, CD4 count, patch quality scores |

**How it works — the counting process format:**  
Each subject contributes multiple rows, one per interval where covariate values are constant. Each row has a `(tstart, tstop]` interval and an event indicator at `tstop`. The `Surv(tstart, tstop, event)` three-argument form handles this.

**Also covered:** time × covariate interactions as a fix for proportional hazards violations (a PH violation is mathematically equivalent to a time-varying coefficient).

---

## Setup

In [1]:
library(tidyverse)
library(survival)    # Surv(), coxph(), survSplit(), tmerge()
library(survminer)
library(broom)

set.seed(42)

# ── Baseline data: one row per patch ─────────────────────────────────────────
# Patches monitored monthly for colonisation; water quality measured each month
n_patches <- 120
n_months  <- 24

baseline <- tibble(
  id      = 1:n_patches,
  habitat = factor(sample(c("restored","degraded"), n_patches,
                          replace=TRUE, prob=c(0.55,0.45)),
                   levels=c("restored","degraded")),
  # True colonisation time
  true_t  = rexp(n_patches,
                 rate=ifelse(habitat=="restored", 0.08, 0.04))
) %>%
  mutate(
    status = as.integer(true_t <= n_months),
    time   = pmin(true_t, n_months)
  )

# ── Monthly water quality measurements per patch ──────────────────────────────
# Water quality measured at months 1, 6, 12, 18 (four measurement occasions)
measure_times <- c(0, 6, 12, 18)

wq_long <- expand_grid(id=1:n_patches, measure_month=measure_times) %>%
  left_join(baseline %>% select(id, habitat), by="id") %>%
  mutate(
    water_qual = rnorm(n(),
      mean = ifelse(habitat=="restored", 6.5, 4.5) + 0.05 * measure_month,
      sd   = 1.2
    )
  )

cat("Baseline: ", n_patches, "patches\n")
cat("Events observed:", sum(baseline$status), "/", n_patches, "\n")

Warning message:
"package 'tidyverse' was built under R version 4.4.3"
Warning message:
"package 'ggplot2' was built under R version 4.4.3"
Warning message:
"package 'purrr' was built under R version 4.4.3"
Warning message:
"package 'dplyr' was built under R version 4.4.3"
Warning message:
"package 'stringr' was built under R version 4.4.3"
── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.0     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.6.0
✔ ggplot2   4.0.2     ✔ tibble    3.2.1
✔ lubridate 1.9.3     ✔ tidyr     1.3.1
✔ purrr     1.2.1     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
Warning message:
"package 'survival' was built under R version 4.4.3"
Warning message:
"package 'survminer' was built under R version

Baseline:  120 patches
Events observed: 79 / 120 


---

## Building the Counting-Process Dataset with `tmerge()`

In [3]:
# ── Step 1: initialise tmerge from baseline data ──────────────────────────────
tv_data <- survival::tmerge(
  data1 = baseline,                            # baseline covariate data
  data2 = baseline,                            # event/censoring time source
  id    = id,
  event = event(time, status)                  # define the outcome
)

# ── Step 2: add time-varying water quality measurements ───────────────────────
tv_data <- survival::tmerge(
  data1 = tv_data,
  data2 = wq_long,
  id    = id,
  water_qual = tdc(measure_month, water_qual)   # tdc = time-dependent covariate
  # water_qual takes the value measured at measure_month,
  # held constant until the next measurement
)

# ── Inspect the result ────────────────────────────────────────────────────────
tv_data %>%
  filter(id %in% 1:3) %>%
  dplyr::select(id, tstart, tstop, event, habitat, water_qual) %>%
  as_tibble() %>%   # convert so n= argument works
  print(n = 20)

cat(sprintf("\nOriginal rows: %d → Expanded rows: %d\n",
            n_patches, nrow(tv_data)))

# A tibble: 9 × 6
     id tstart tstop event habitat  water_qual
  <int>  <dbl> <dbl> <int> <fct>         <dbl>
1     1      0  6        0 degraded       3.21
2     1      6 12        0 degraded       5.00
3     1     12 18        0 degraded       4.66
4     1     18 24        0 degraded       6.11
5     2      0  6        0 degraded       6.22
6     2      6 12        0 degraded       3.61
7     2     12 18        0 degraded       5.65
8     2     18 24        0 degraded       5.50
9     3      0  1.84     1 restored       7.57

Original rows: 120 → Expanded rows: 332


---

## Cox Model with Time-Varying Covariates

In [4]:
# ── Model 1: baseline covariates only (ignores water quality changes) ─────────
cox_baseline <- coxph(
  Surv(time, status) ~ habitat + water_qual,
  data = baseline %>% left_join(
    wq_long %>% filter(measure_month==0) %>% select(id, water_qual),
    by="id")
)

# ── Model 2: time-varying water quality ───────────────────────────────────────
cox_tv <- coxph(
  Surv(tstart, tstop, event) ~ habitat + water_qual,   # three-argument Surv()
  data    = tv_data,
  cluster = id    # robust SEs: accounts for multiple rows per subject
)
summary(cox_tv)

# ── Compare coefficient estimates ────────────────────────────────────────────
tibble(
  term             = c("habitatdegraded","water_qual"),
  baseline_HR      = round(exp(coef(cox_baseline)), 3),
  tv_HR            = round(exp(coef(cox_tv)), 3)
) %>% print()
# Differences reflect confounding by time-varying water quality:
# the baseline model attributes to 'habitat' some variation that is
# actually due to water quality changes over time

Call:
coxph(formula = Surv(tstart, tstop, event) ~ habitat + water_qual, 
    data = tv_data, cluster = id)

  n= 332, number of events= 79 

                    coef exp(coef) se(coef) robust se      z Pr(>|z|)   
habitatdegraded -0.76769   0.46408  0.29615   0.28659 -2.679  0.00739 **
water_qual       0.03387   1.03445  0.09870   0.10661  0.318  0.75069   
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

                exp(coef) exp(-coef) lower .95 upper .95
habitatdegraded    0.4641     2.1548    0.2646    0.8138
water_qual         1.0345     0.9667    0.8394    1.2748

Concordance= 0.604  (se = 0.035 )
Likelihood ratio test= 13.38  on 2 df,   p=0.001
Wald test            = 13.32  on 2 df,   p=0.001
Score (logrank) test = 13.69  on 2 df,   p=0.001,   Robust = 13.74  p=0.001

  (Note: the likelihood ratio and score tests assume independence of
     observations within a cluster, the Wald and robust score tests do not).

# A tibble: 2 × 3
  term            baseline_HR tv_HR
  <chr>                 <dbl> <dbl>
1 habitatdegraded       0.326 0.464
2 water_qual            0.876 1.03 


---

## Time-Varying Coefficients: Fixing PH Violations

A proportional hazards violation means the effect of a covariate changes with time — this is mathematically a time-varying coefficient. Two approaches: `tt()` interaction and `survSplit()`.

In [5]:
# ── Simulate a PH violation: habitat effect declines over time ────────────────
n <- 200
ph_viol_data <- tibble(
  id      = 1:n,
  habitat = factor(sample(c("restored","degraded"), n, replace=TRUE),
                   levels=c("restored","degraded")),
  # Hazard ratio for degraded starts high but declines with log(t)
  true_t  = rexp(n, rate = ifelse(habitat=="restored", 0.08,
                                   0.12 * exp(-0.04 * rexp(n, 0.1))))
) %>%
  mutate(status=as.integer(true_t<=30), time=pmin(true_t,30))

# ── Check PH assumption ───────────────────────────────────────────────────────
cox_ph <- coxph(Surv(time,status) ~ habitat, data=ph_viol_data)
ph_test <- cox.zph(cox_ph)
print(ph_test)   # expect significant violation

# ── Approach 1: tt() — time-transformed covariate interaction ─────────────────
cox_tt <- coxph(
  Surv(time, status) ~ habitat + tt(habitat),
  data = ph_viol_data,
  tt   = function(x, t, ...) (x=="degraded") * log(t + 0.5)
  # coefficient of tt(habitat) tests whether the effect changes with log(t)
)
summary(cox_tt)

# ── Approach 2: piecewise — split at a clinical break point ───────────────────
# E.g. early period (t<=10) vs. late period (t>10)
split_data <- survival::survSplit(
  Surv(time, status) ~ .,
  data   = ph_viol_data,
  cut    = 10,           # split point
  episode = "period"     # new variable coding early (1) vs. late (2)
)

cox_piece <- coxph(
  Surv(tstart, time, status) ~ habitat : strata(period),
  data = split_data
)
summary(cox_piece)
# Separate HR for habitat in early vs. late period
# If early HR ≠ late HR → confirms PH violation was real

        chisq df    p
habitat 0.385  1 0.54
GLOBAL  0.385  1 0.54


Call:
coxph(formula = Surv(time, status) ~ habitat + tt(habitat), data = ph_viol_data, 
    tt = function(x, t, ...) (x == "degraded") * log(t + 0.5))

  n= 200, number of events= 180 

                    coef exp(coef) se(coef)      z Pr(>|z|)
habitatdegraded -0.42285   0.65518  0.36099 -1.171    0.241
tt(habitat)      0.07458   1.07744  0.16891  0.442    0.659

                exp(coef) exp(-coef) lower .95 upper .95
habitatdegraded    0.6552     1.5263    0.3229     1.329
tt(habitat)        1.0774     0.9281    0.7738     1.500

Concordance= 0.54  (se = 0.02 )
Likelihood ratio test= 3.64  on 2 df,   p=0.2
Wald test            = 3.58  on 2 df,   p=0.2
Score (logrank) test = 3.61  on 2 df,   p=0.2


Call:
coxph(formula = Surv(tstart, time, status) ~ habitat:strata(period), 
    data = split_data)

  n= 281, number of events= 180 

                                          coef exp(coef) se(coef)     z
habitatrestored:strata(period)period=1 0.42059   1.52286  0.18921 2.223
habitatdegraded:strata(period)period=1      NA        NA  0.00000    NA
habitatrestored:strata(period)period=2 0.01267   1.01275  0.25790 0.049
habitatdegraded:strata(period)period=2      NA        NA  0.00000    NA
                                       Pr(>|z|)  
habitatrestored:strata(period)period=1   0.0262 *
habitatdegraded:strata(period)period=1       NA  
habitatrestored:strata(period)period=2   0.9608  
habitatdegraded:strata(period)period=2       NA  
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

                                       exp(coef) exp(-coef) lower .95 upper .95
habitatrestored:strata(period)period=1     1.523     0.6567    1.0510     2.207
habitatdegraded:strata(perio

---

## Landmark Analysis: A Simpler Alternative

In [6]:
# Landmark analysis: restrict analysis to subjects still event-free at a
# landmark time, then update covariate values at that time point.
# Simpler than full time-varying Cox; avoids immortal time bias.

landmark_time <- 12   # month 12 landmark

# Subjects still event-free at landmark
survivors <- baseline %>%
  filter(time > landmark_time) %>%
  mutate(
    # Remaining follow-up from landmark
    time_from_lm = time - landmark_time,
    status_lm    = status
  )

# Water quality at landmark (month 12)
wq_at_lm <- wq_long %>% filter(measure_month == 12)

landmark_data <- survivors %>%
  left_join(wq_at_lm %>% select(id, water_qual), by="id")

cox_lm <- coxph(
  Surv(time_from_lm, status_lm) ~ habitat + water_qual,
  data = landmark_data
)
cat(sprintf("Landmark analysis (n=%d at-risk at month %d):\n",
            nrow(landmark_data), landmark_time))
print(broom::tidy(cox_lm, exponentiate=TRUE, conf.int=TRUE))

Landmark analysis (n=70 at-risk at month 12):
# A tibble: 2 × 7
  term            estimate std.error statistic p.value conf.low conf.high
  <chr>              <dbl>     <dbl>     <dbl>   <dbl>    <dbl>     <dbl>
1 habitatdegraded    0.538     0.460    -1.35    0.177    0.218      1.32
2 water_qual         1.05      0.176     0.297   0.766    0.746      1.49


---

## Common Pitfalls

**1. Using last-observation-carried-forward (LOCF) instead of tmerge**  
Carrying the most recent measurement forward as a fixed baseline value is not the same as correctly modelling a time-varying covariate. It underestimates variation and can introduce bias if the covariate trajectory is related to the event.

**2. Immortal time bias**  
If subjects must survive to receive a treatment or measurement, that survival time before exposure is 'immortal' — subjects cannot have the event during it. Ignoring immortal time inflates the apparent benefit of the treatment. Use `tmerge()` to correctly assign covariate values only from the time they became applicable.

**3. Forgetting `cluster = id` in `coxph()`**  
When subjects contribute multiple rows, standard errors from `coxph()` assume independent observations. The `cluster` argument applies a robust sandwich estimator to correct for within-subject correlation across rows.

**4. Interpreting a time-varying covariate effect as causal**  
Internal time-varying covariates (e.g., a biomarker that rises before the event) can produce spurious associations because the covariate may be on the causal pathway or a symptom of the upcoming event. Treat such associations with caution and consider marginal structural models for causal questions.

**5. Using landmark analysis when the landmark time is data-driven**  
Choosing the landmark time based on where survival curves diverge or converge is post-hoc selection and inflates the apparent difference. Landmark time should be specified a priori based on clinical or biological rationale.

---
*r_methods_library · Samantha McGarrigle · [github.com/samantha-mcgarrigle](https://github.com/samantha-mcgarrigle)*